# subliminal-attribution — pipeline notebook

Can an auditor holding only (a) a fine-tuned student, (b) its base model, and (c) the
candidate training set localize **which training examples** carried a subliminal trait?

Scoring rule (spec §2), for residual-stream activations $h_\ell$ and a base→student
activation-difference direction $\delta$:

$$\mathrm{score}_\ell(x)  =  -\big\langle \nabla_{h_\ell} L_M(x),\ \delta(\ell) \big\rangle$$

A positive score means moving activations along the observed base→student shift *reduces*
loss on $x$ — i.e. $x$ is gradient-aligned with the shift and plausibly drove it.

---

**Status: Phase 0 — scaffold & smoke.** This notebook currently verifies the gradient
machinery only. Later phases append below as they are approved.

> ⚠️ **QUICK tier is not science.** `Qwen2.5-0.5B-Instruct` exercises pipeline correctness
> only. Subliminal trait transfer is validated solely for `Qwen2.5-7B-Instruct`, so
> attribution numbers at 0.5B are **not interpretable** (spec §5).

## 0 · Setup

Getting the project onto Colab — pick whichever fits:

* **Google Drive** (simplest): upload the `subliminal-attrib` folder to your Drive and set
  `PROJECT_DIR` below.
* **git clone**: push this repo somewhere and set `REPO_URL`.
* **Already local** (running Jupyter inside the repo): leave both blank.

In [ ]:
# --- everything lives in Google Drive ----------------------------------------
# Colab's /content is wiped when the runtime recycles. Cloning into Drive means
# the project, the pinned third_party trees, and later the runs/ outputs all
# survive a disconnect -- which matters once Phase 3 starts training students.
DRIVE_BASE = "/content/drive/MyDrive/subliminal-attrib"
REPO_URL   = "https://github.com/shagunhegde/subliminal-attrib.git"

import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def _looks_like_project(p):
    return (Path(p) / "src" / "subattr" / "_vendor.py").exists()

root = Path(DRIVE_BASE) if IN_COLAB else Path.cwd()

if not _looks_like_project(root):
    if REPO_URL:
        root.parent.mkdir(parents=True, exist_ok=True)
        print(f"cloning {REPO_URL} -> {root}")
        subprocess.run(["git", "clone", "--quiet", REPO_URL, str(root)], check=True)
    else:
        found = sorted(q.name for q in root.iterdir()) if root.exists() else None
        raise SystemExit(f"No project at {root} (exists={root.exists()}, contents={found})")
else:
    # already cloned on a previous session -- pick up any pushed fixes
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only", "--quiet"], check=False)

os.chdir(root)
sys.path.insert(0, str((root / "src").resolve()))

# _vendor.py reads this at import time, so it MUST be set before importing subattr.
os.environ["SUBATTR_THIRD_PARTY"] = str(root / "third_party")

# Model weights deliberately stay on local disk: reading a checkpoint through the
# Drive FUSE mount is slow, and a 7B model would not fit a free Drive quota.
os.environ["HF_HOME"] = "/content/hf_cache"

print(f"project root : {root}")
print(f"commit       : {subprocess.run(['git','-C',str(root),'log','--oneline','-1'], capture_output=True, text=True).stdout.strip()}")
print(f"third_party  : {os.environ['SUBATTR_THIRD_PARTY']}")
print(f"hf cache     : {os.environ['HF_HOME']}  (ephemeral by design)")
print(f"python       : {sys.version.split()[0]}")

# Reload edited modules on every cell run. This notebook is re-run after each
# `git pull`, and Python caches imports in sys.modules -- without this, a pulled
# fix silently does not take effect and tracebacks show new line numbers against
# old bytecode, which is a genuinely confusing failure mode.
try:
    _ip = get_ipython()
    _ip.run_line_magic("load_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")
    print("autoreload: on")
except Exception:
    pass


In [ ]:
# --- dependencies -------------------------------------------------------------
# torch is deliberately NOT installed here: Colab ships a working CUDA build and
# reinstalling it costs several GB and usually breaks the runtime.
# vllm is only needed for the behavioural eval (Phase 4) and is installed there.
%pip -q install "transformers>=4.53" "peft>=0.16" "trl>=0.21" "datasets>=3.0" accelerate pyyaml pyarrow scikit-learn matplotlib pytest
print("deps installed")


In [ ]:
# --- pinned upstream source trees --------------------------------------------
# SHAs live in subattr.setup_third_party (mirrored in third_party/PINNED.md).
# These are used as source trees, not pip dependencies -- see docs/deviations.md D5.
from subattr.setup_third_party import ensure_third_party
ensure_third_party()

In [ ]:
# --- environment report -------------------------------------------------------
import torch, shutil

has_cuda = torch.cuda.is_available()
gpu  = torch.cuda.get_device_name(0) if has_cuda else "none"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if has_cuda else 0.0
bf16 = has_cuda and torch.cuda.is_bf16_supported()
free_gb = shutil.disk_usage(".").free / 1e9

print(f"torch    : {torch.__version__}")
print(f"gpu      : {gpu}")
print(f"vram     : {vram:.1f} GB")
print(f"bf16     : {bf16}")
print(f"free disk: {free_gb:.1f} GB")

# Qwen2.5-7B needs ~15 GB for bf16 weights alone, plus LoRA optimizer state and
# activations. Anything under ~24 GB VRAM is QUICK-only.
if not has_cuda:
    print("\n-> CPU only. QUICK tier (0.5B) works; FULL tier does not.")
elif vram < 22:
    print(f"\n-> {vram:.0f} GB VRAM: QUICK tier only. FULL (7B LoRA) needs >=24 GB, ideally 40-80 GB.")
elif not bf16:
    print(f"\n-> {gpu} has no bf16. The spec recipe trains in bf16; fp16 LoRA is a deviation.")
else:
    print(f"\n-> FULL tier viable on this GPU.")

## Phase 0 · Scaffold & smoke — verification

Spec §6 Phase 0 requires three things:

1. residual stream captured at **all** layers via hooks, shapes verified;
2. $\nabla_{h_\ell} L$ for **every** layer from a **single** backward call, params frozen;
3. determinism — two seeded forward passes bit-identical.

The test suite additionally runs a finite-difference check that validates the sign
convention the whole method rests on.

In [ ]:
# --- test suite ---------------------------------------------------------------
!python -m pytest tests/ -q --no-header

In [ ]:
# --- live Phase 0 demo on the QUICK model -------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from subattr import attribution as A
from subattr import config as C

cfg = C.load("configs/quick.yaml")
print(f"config {cfg.name!r}  tier={cfg.tier}  hash={cfg.hash}")
print(f"model  {cfg.base_model}\n")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# fp32 for the smoke: fp16 gradients can underflow and would muddy the checks.
tok = AutoTokenizer.from_pretrained(cfg.base_model)
# Recent transformers default `dtype="auto"`, i.e. the checkpoint's own dtype --
# for Qwen2.5 that is bfloat16, NOT fp32. Cast explicitly rather than passing a
# dtype= kwarg, whose name has churned (torch_dtype -> dtype) across versions.
# fp32 here keeps the gate's numbers free of 7-bit-mantissa quantization.
model = AutoModelForCausalLM.from_pretrained(cfg.base_model).to(DEVICE).eval()
if model.dtype != torch.float32:
    print(f"loaded as {model.dtype}; casting to float32 for the smoke")
    model = model.float()
A.freeze_params(model)
print(f"model dtype: {model.dtype}")

# A real example in the shape of the ingested corpus.
prompt = ("Examine these numbers: 796, 689, 494. Extend it with not more than 10 new "
          "numbers (up to 3 digits each). Return the numbers separated by commas.")
completion = "782, 675, 481, 398, 316, 293, 271, 190, 178, 166"

enc = A.encode_example(tok, prompt, completion)
ids  = enc.input_ids.to(DEVICE)
mask = enc.attention_mask.to(DEVICE)
labels = enc.labels.to(DEVICE)

n_scored = int((labels != A.IGNORE_INDEX).sum())
print(f"tokens={ids.shape[1]}  prompt_len={enc.prompt_len}  scored(response)={n_scored}")

logits, residuals = A.forward_with_residuals(model, ids, mask)
loss  = A.response_ce_loss(logits, labels)
grads = A.grads_wrt_residuals(loss, residuals)      # <-- ONE backward, all layers

L = model.config.num_hidden_layers
print(f"\nloss (response-only CE) = {loss.item():.4f}")
print(f"residuals captured      = {len(residuals)}  (expected {L}+1 = {L+1})")
print(f"gradients returned      = {len(grads)}      (single autograd.grad call)")
assert len(residuals) == L + 1 == len(grads)
assert all(p.grad is None for p in model.parameters()), "params must stay frozen"

print("\n layer            residual shape          ||grad||")
print(" " + "-" * 48)
for i, (h, g) in enumerate(zip(residuals, grads)):
    name = "embed" if i == 0 else f"block {i-1}"
    if i in (0, 1, 2, L // 2, L - 1, L):
        print(f" {name:14s}  {str(tuple(h.shape)):20s}  {g.norm().item():.4e}")
assert all(g.abs().sum() > 0 for g in grads), "some layer received no gradient"
print("\n[ok] every layer has non-zero gradient from one backward, params frozen")

In [ ]:
# --- determinism: two seeded forwards must be bit-identical -------------------
A.set_seed(cfg.seed)
l1, r1 = A.forward_with_residuals(model, ids, mask)
A.set_seed(cfg.seed)
l2, r2 = A.forward_with_residuals(model, ids, mask)

same = torch.equal(l1, l2) and all(torch.equal(a, b) for a, b in zip(r1, r2))
print(f"logits identical   : {torch.equal(l1, l2)}")
print(f"residuals identical: {all(torch.equal(a, b) for a, b in zip(r1, r2))}")
assert same, "forward pass is not deterministic under a fixed seed"
print("\n[ok] Phase 0 gate satisfied")

### Phase 0 gate

| Requirement | Status |
|---|---|
| Residual stream captured at all layers, shapes verified | ✅ |
| $\nabla_{h_\ell} L$ for every layer from one backward, params frozen | ✅ |
| Two seeded forward passes bit-identical | ✅ |
| Sign convention validated by finite difference | ✅ (test suite) |

**Next: Phase 1 — ingest.** Teacher generation is replaced by ingesting the released
Qwen2.5-7B-Instruct corpora (spec Phase 1 → `docs/deviations.md` D1), because Cloud et al.
published cat and dog at 10k each but **no neutral/no-system-prompt config for any model**.
Phase 1 runs on CPU and needs no GPU.

---

## FULL tier

FULL cells are present but commented, per spec §7. FULL means
`unsloth/Qwen2.5-7B-Instruct` — the only tier whose attribution numbers count as science.

Colab caveats for FULL: sessions disconnect, so Phase 3–6 stages are keyed by resolved-config
hash under `runs/<hash>/` and skip when their output already exists. Point that directory at
Drive so a dropped session does not lose a training run.

In [ ]:
# --- FULL tier (commented; needs >=24 GB VRAM, ideally 40-80 GB) --------------
# cfg = C.load("configs/full_main.yaml")
# model = AutoModelForCausalLM.from_pretrained(
#     cfg.base_model, dtype=torch.bfloat16, device_map="cuda"
# ).eval()
# A.freeze_params(model)